In [1]:
%pip install -U langchain-ollama langchain-chroma chromadb langchain-core

  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/579.8 kB ? eta -:--:--
   ------------------ --------------------- 262.1/579.8 kB ? eta -:--:--
   ------------------ --------------------- 262.1/579.8 kB ? eta -:--:--
   ------------------ --------------------- 262.1/579.8 kB ? eta -:-


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [3]:
doc1 = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
    metadata={"team": "Royal Challengers Bangalore"}
)
doc2 = Document(
    page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He is known for his calm demeanor and ability to play big innings under pressure.",
    metadata={"team": "Mumbai Indians"}
)
doc3 = Document(
    page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
    metadata={"team": "Chennai Super Kings"}
)
doc4 = Document(
    page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
    metadata={"team": "Mumbai Indians"}
)
doc5 = Document(
    page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
    metadata={"team": "Chennai Super Kings"}
)

In [4]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [5]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vector_store = Chroma(
    collection_name="sample_ollama",
    embedding_function=embeddings,
    persist_directory="my_chroma_db_ollama"
)

In [7]:
doc_ids = [f"ipl-player-{i}" for i in range(1, len(docs) + 1)]
added_ids = vector_store.add_documents(documents=docs, ids=doc_ids)
print("Added document IDs:", added_ids)

Added document IDs: ['ipl-player-1', 'ipl-player-2', 'ipl-player-3', 'ipl-player-4', 'ipl-player-5']


In [8]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['ipl-player-1',
  'ipl-player-2',
  'ipl-player-3',
  'ipl-player-4',
  'ipl-player-5'],
 'embeddings': array([[-0.024059  ,  0.04453161, -0.18949507, ..., -0.01605532,
         -0.05859691,  0.0025295 ],
        [-0.03270196,  0.07475892, -0.163626  , ..., -0.06488526,
         -0.02263143, -0.01309493],
        [ 0.02280271,  0.0657957 , -0.18388067, ..., -0.021293  ,
         -0.04229931,  0.00040438],
        [ 0.01442985,  0.03144638, -0.18336748, ..., -0.02951371,
         -0.03739943,  0.00836507],
        [-0.03468144,  0.03123768, -0.16423446, ..., -0.0237606 ,
         -0.05102901,  0.03770281]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  'Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He is known for his calm demeanor and abili

In [9]:
results = vector_store.similarity_search(
    query="Who among these are a bowler?",
    k=2
)

for doc in results:
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)
    print("-" * 50)

Content: Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.
Metadata: {'team': 'Mumbai Indians'}
--------------------------------------------------
Content: Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
Metadata: {'team': 'Royal Challengers Bangalore'}
--------------------------------------------------


In [10]:
results_with_scores = vector_store.similarity_search_with_score(
    query="Who among these are a bowler?",
    k=2
)

for doc, score in results_with_scores:
    print("Content:", doc.page_content)
    print("Distance score:", score)
    print("Metadata:", doc.metadata)
    print("-" * 50)

Content: Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.
Distance score: 0.6737993359565735
Metadata: {'team': 'Mumbai Indians'}
--------------------------------------------------
Content: Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
Distance score: 0.8671941757202148
Metadata: {'team': 'Royal Challengers Bangalore'}
--------------------------------------------------


In [11]:
vector_store.similarity_search_with_score(
    query="Players who represent Chennai Super Kings",
    k=5,
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='ipl-player-5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.5175581574440002),
 (Document(id='ipl-player-3', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6958295106887817)]

In [12]:
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(
    document_id=doc_ids[0],
    document=updated_doc1
)

In [13]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['ipl-player-1',
  'ipl-player-2',
  'ipl-player-3',
  'ipl-player-4',
  'ipl-player-5'],
 'embeddings': array([[ 0.00654565,  0.04033578, -0.19677536, ..., -0.03231447,
         -0.04001838,  0.00946409],
        [-0.03270196,  0.07475892, -0.163626  , ..., -0.06488526,
         -0.02263143, -0.01309493],
        [ 0.02280271,  0.0657957 , -0.18388067, ..., -0.021293  ,
         -0.04229931,  0.00040438],
        [ 0.01442985,  0.03144638, -0.18336748, ..., -0.02951371,
         -0.03739943,  0.00836507],
        [-0.03468144,  0.03123768, -0.16423446, ..., -0.0237606 ,
         -0.05102901,  0.03770281]], shape=(5, 768)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a 

In [14]:
vector_store.delete(ids=[doc_ids[0]])

In [15]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['ipl-player-2', 'ipl-player-3', 'ipl-player-4', 'ipl-player-5'],
 'embeddings': array([[-0.03270196,  0.07475892, -0.163626  , ..., -0.06488526,
         -0.02263143, -0.01309493],
        [ 0.02280271,  0.0657957 , -0.18388067, ..., -0.021293  ,
         -0.04229931,  0.00040438],
        [ 0.01442985,  0.03144638, -0.18336748, ..., -0.02951371,
         -0.03739943,  0.00836507],
        [-0.03468144,  0.03123768, -0.16423446, ..., -0.0237606 ,
         -0.05102901,  0.03770281]], shape=(4, 768)),
 'documents': ['Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He is known for his calm demeanor and ability to play big innings under pressure.',
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for 